# 03 - Train Probabilistic Circuit

Builds HCLT from a Chow-Liu tree over the selected SAE features, trains with Adam (gradient-based MLE), and calibrates the decision threshold. Saves the monitor.

**Anomaly score = NLL only.** The composite refusal term was removed (it mixed incomparable scales); refusal features are kept solely for interpretability via `explain()`.

**Time estimate:** 15-40 min on GPU.

In [ ]:
import os
import sys
REPO_DIR = '/kaggle/working/pc-sae'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from src.utils import setup_hf_auth
setup_hf_auth()

In [ ]:
!python scripts/03_train_pc.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.utils import METRICS, load_json

training = load_json(METRICS / 'pc_training.json')
print(f"layer: {training['layer']}")
print(f"n features: {training['n_features']}")
print(f"train NLL: {training['train_nll']:.3f}")
print(f"val NLL: {training['val_nll']:.3f}")
print(f"unsafe NLL: {training['unsafe_nll']:.3f}")
print(f"separation: {training['separation']:.3f}")
print(f"\ncalibration AUROC: {training['calibration']['auroc']:.4f}")
print(f"threshold tau: {training['calibration']['threshold']:.3f}")
print(f"TPR at threshold: {training['calibration']['tpr_at_threshold']:.3f}")

In [ ]:
history = training['history']
epochs = [h['epoch'] for h in history]
train_nll = [h['train_nll'] for h in history]
val_nll = [h.get('val_nll') for h in history if 'val_nll' in h]

plt.figure(figsize=(10, 5))
plt.plot(epochs, train_nll, label='train', linewidth=2)
if val_nll:
    plt.plot(epochs[:len(val_nll)], val_nll, label='val', linewidth=2)
plt.xlabel('epoch'); plt.ylabel('NLL'); plt.legend(fontsize=12)
plt.title('PC training curves')
plt.grid(alpha=0.3)
plt.savefig('outputs/results/figures/pc_training.png', dpi=100)
plt.show()

In [ ]:
import torch
from src.pc import HCLT
from src.utils import PC_DIR, SAE_FEATURES, load_npz, load_json

decision = load_json(METRICS / 'localization_decision.json')
layer = decision['best_layer']
feat_map = np.array(decision['selected_feature_indices'], dtype=np.int64)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pc = HCLT.load(PC_DIR / 'monitor' / 'pc', device=device)
z_safe = load_npz(SAE_FEATURES / f'layer_{layer}_safe.npz')['z_binary']
z_unsafe = load_npz(SAE_FEATURES / f'layer_{layer}_unsafe.npz')['z_binary']
X_safe = z_safe[:, feat_map].astype(np.float32)
X_unsafe = z_unsafe[:, feat_map].astype(np.float32)

nll_safe = -pc.log_prob(X_safe).detach().cpu().numpy()
nll_unsafe = -pc.log_prob(X_unsafe).detach().cpu().numpy()
print(f'NLL safe: mean={nll_safe.mean():.3f} std={nll_safe.std():.3f}')
print(f'NLL unsafe: mean={nll_unsafe.mean():.3f} std={nll_unsafe.std():.3f}')

plt.figure(figsize=(10, 5))
plt.hist(nll_safe, bins=50, alpha=0.6, label='safe', density=True, color='green')
plt.hist(nll_unsafe, bins=50, alpha=0.6, label='unsafe', density=True, color='red')
tau = training['calibration']['threshold']
plt.axvline(tau, color='black', linestyle='--', label=f'threshold ({tau:.2f})')
plt.xlabel('NLL'); plt.ylabel('density'); plt.legend()
plt.title('NLL distribution (safe vs unsafe)')
plt.grid(alpha=0.3)
plt.savefig('outputs/results/figures/nll_distribution.png', dpi=100)
plt.show()

In [ ]:
!cat outputs/logs/train_pc.log 2>/dev/null | tail -40
print('\n--- ERRORS ---')
!cat outputs/logs/errors.log 2>/dev/null | tail -20